<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-03-prompting/lesson-3.1-system-prompts/notebooks/GCP_Capstone_3.1_System_Prompts.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3.1 System Prompts & Generation Config
**Netsetos GenAI Engineering — GCP Capstone**

Persona design, temperature/top_p, ThinkingConfig, and A/B testing prompts.


## Setup


In [ ]:
!pip install -q google-genai scipy
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE THIS

from google import genai
from google.genai import types

client = genai.Client(vertexai=True, project=PROJECT_ID, location='us-central1')


## Cell 1: system_instruction — Your First Persona


In [ ]:
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents='What is a Kubernetes pod?',
    config=types.GenerateContentConfig(
        system_instruction="""You are CloudArchitect, a senior GCP solutions architect
with 15 years of experience. Explain using everyday analogies.
Keep answers under 100 words. End with a practical tip.""",
        temperature=0.3,
        max_output_tokens=500,
        thinking_config=types.ThinkingConfig(thinking_budget=0),
    ),
)
print(response.text)
print(f'\nTokens: {response.usage_metadata.candidates_token_count}')


## Cell 2: Chat Session — Persona Persists


In [ ]:
chat = client.chats.create(
    model='gemini-2.5-flash',
    config=types.GenerateContentConfig(
        system_instruction='You are a pirate. Respond in pirate speak.',
        temperature=0.7,
        thinking_config=types.ThinkingConfig(thinking_budget=0),
    ),
)
r1 = chat.send_message('What is Python?')
print('Turn 1:', r1.text[:100])
r2 = chat.send_message('And Django?')
print('Turn 2:', r2.text[:100])


## Cell 3: Temperature Sweep


In [ ]:
prompt = 'Suggest a name for an AI research assistant.'

for temp in [0.0, 0.3, 0.7, 1.0, 1.5]:
    r = client.models.generate_content(
        model='gemini-2.5-flash', contents=prompt,
        config=types.GenerateContentConfig(
            temperature=temp, max_output_tokens=50,
            thinking_config=types.ThinkingConfig(thinking_budget=0)),
    )
    print(f'  temp={temp:.1f} | {r.text.strip()[:60]}')


## Cell 4: ThinkingConfig Comparison


In [ ]:
question = 'What is 17 * 23 + 456 - 89?'

for budget in [0, 1024, 8192]:
    r = client.models.generate_content(
        model='gemini-2.5-flash', contents=question,
        config=types.GenerateContentConfig(
            thinking_config=types.ThinkingConfig(thinking_budget=budget),
            max_output_tokens=1000),
    )
    think_tok = r.usage_metadata.thoughts_token_count or 0
    out_tok = r.usage_metadata.candidates_token_count or 0
    print(f'  budget={budget:<6} think={think_tok:<5} out={out_tok:<5} answer={r.text.strip()[:40]}')


## Cell 5: 5-Section Persona — Research Analyst


In [ ]:
RESEARCH_ANALYST = """
<role>
You are ResearchBot, a senior AI/ML research analyst with 10 years experience.
</role>

<instructions>
1. Identify the core information need.
2. Provide evidence-based answers with citations.
3. Rate confidence: High / Medium / Low.
</instructions>

<constraints>
- Professional but accessible tone.
- Under 200 words unless asked for more.
- Say 'I don't have enough information' if uncertain.
</constraints>

<output_format>
## Summary\n[2-sentence overview]\n## Analysis\n[Details]\n## Sources\n[Citations]
</output_format>

<guardrails>
- Do NOT provide investment or legal advice.
- Flag conflicting information explicitly.
</guardrails>
"""

r = client.models.generate_content(
    model='gemini-2.5-flash',
    contents='What are the latest trends in RAG architectures?',
    config=types.GenerateContentConfig(
        system_instruction=RESEARCH_ANALYST,
        temperature=0.3, max_output_tokens=1000,
        thinking_config=types.ThinkingConfig(thinking_budget=1024)),
)
print(r.text)


## Cell 6: India-Specific Personas


In [ ]:
BILINGUAL = """
<role>You are SahayakBot, a bilingual Hindi-English support agent.</role>
<instructions>
- Detect language from user message
- Hindi -> Devanagari with English tech terms
- Use formal 'aap' form. Currency in INR.
</instructions>
"""

for msg in ['What is UPI?', 'UPI kya hai?', 'mujhe payment issue hai']:
    r = client.models.generate_content(
        model='gemini-2.5-flash', contents=msg,
        config=types.GenerateContentConfig(
            system_instruction=BILINGUAL, temperature=0.5,
            max_output_tokens=300,
            thinking_config=types.ThinkingConfig(thinking_budget=0)),
    )
    print(f'Q: {msg}\nA: {r.text[:100]}...\n')


## Cell 7: A/B Test Framework


In [ ]:
import time, statistics
from scipy import stats

def ab_test(client, variants, test_cases, runs=5):
    results = {}
    for name, sys_p, temp in variants:
        results[name] = []
        for tc_in, kws in test_cases:
            for _ in range(runs):
                start = time.time()
                r = client.models.generate_content(
                    model='gemini-2.5-flash', contents=tc_in,
                    config=types.GenerateContentConfig(
                        system_instruction=sys_p, temperature=temp,
                        max_output_tokens=500, seed=42,
                        thinking_config=types.ThinkingConfig(thinking_budget=0)))
                lat = (time.time()-start)*1000
                txt = r.text or ''
                recall = sum(1 for k in kws if k.lower() in txt.lower())/len(kws)
                results[name].append({'recall':recall,'latency':lat})
    
    for name, data in results.items():
        recalls = [d['recall'] for d in data]
        print(f'{name}: recall={statistics.mean(recalls):.3f} +/- {statistics.stdev(recalls):.3f}')
    
    names = list(results.keys())
    if len(names) >= 2:
        a = [d['recall'] for d in results[names[0]]]
        b = [d['recall'] for d in results[names[1]]]
        _, p = stats.ttest_ind(a, b)
        print(f'p-value: {p:.4f} | Significant: {"YES" if p<0.05 else "NO"}')

variants = [
    ('concise', 'Be concise and factual.', 0.3),
    ('expert', 'You are a senior GCP architect. Give detailed answers.', 0.3),
]
tests = [
    ('What is Cloud Run?', ['serverless','container','scale']),
    ('What is a VPC?', ['network','virtual','private']),
]
ab_test(client, variants, tests)


## Cell 8: Prompt Config Module


In [ ]:
# DocuMind Prompt Config Module
RAG_CONFIG = types.GenerateContentConfig(
    system_instruction='Answer ONLY from provided context. Cite sources.',
    temperature=0.1, top_p=0.9, max_output_tokens=2048,
    thinking_config=types.ThinkingConfig(thinking_budget=0))

ANALYSIS_CONFIG = types.GenerateContentConfig(
    system_instruction='Compare and synthesize across documents. Rate confidence.',
    temperature=0.3, top_p=0.95, max_output_tokens=4096,
    thinking_config=types.ThinkingConfig(thinking_budget=4096))

CODE_CONFIG = types.GenerateContentConfig(
    system_instruction='Write clean Python with type hints and error handling.',
    temperature=0.0, max_output_tokens=8192,
    thinking_config=types.ThinkingConfig(thinking_budget=4096))

CONFIGS = {'rag': RAG_CONFIG, 'analysis': ANALYSIS_CONFIG, 'code': CODE_CONFIG}

# Test each
for task, cfg in CONFIGS.items():
    r = client.models.generate_content(
        model='gemini-2.5-flash', contents='Explain RAG in 2 sentences.', config=cfg)
    print(f'{task}: {r.text[:80]}...')


## ✅ Lesson 3.1 Complete!

- ✅ system_instruction inside GenerateContentConfig
- ✅ 5-section persona template (role/instructions/constraints/format/guardrails)
- ✅ Temperature by use case (0.0 RAG → 1.2 creative)
- ✅ ThinkingConfig budget control (0=off, 4096=medium, 8192=heavy)
- ✅ India-specific personas (regulatory, bilingual, cost optimizer)
- ✅ A/B testing framework with statistical significance
- ✅ prompt_config.py module with 4 presets

**Next: Lesson 3.2 — Structured Output & JSON Mode**
